In [11]:
import json 
import os  
import pandas as pd

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "poti2010searching")
original_data_pathway = os.path.join(pathway, "original_data")
original_sub_pathway = os.path.join(original_data_pathway, "EVApe_landmark_data_PKanngiesser")

starting_point= original_sub_pathway

temp = []

for dirpath, dirnames, filenames in os.walk(starting_point):
    for index, filename in enumerate([f for f in filenames if f.endswith("summary.csv")]):
        sav_filepath = os.path.join(dirpath, filename)
        # print(sav_filepath)
        x = pd.read_csv(sav_filepath)
        x.columns = map(str.lower, x.columns)
        x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
        x['file_name']= filename

        temp.append(x)

fulldf = pd.concat(temp, ignore_index=True, sort=False)


comp_out_path_stand = os.path.join(original_data_pathway, 'poti_full_summary.csv')
fulldf.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

complete_path_1 = os.path.join(original_data_pathway, "poti_full_summary.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [12]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)
df['study_id']='poti2010searching'

# df.columns = df.columns.str.replace(". ", "_",  regex=True)
# df.columns

In [13]:
df.rename(columns={"condition.1": "condition_1",
   'name.1':'name_1',
   'name':'ape',
   'experiment':'experiment_name'}, inplace=True)

In [14]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
# df['experiment_name'].unique()

In [15]:
code_list=["experiment_name"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '4 landmarks':
            entry = "1b"
        elif entry =='2 landmarks 1 hole':
            entry = "2b"
        elif entry =='2 landmark 3 holes':
            entry = "2c"
        elif entry =='2 landmarks vd':
            entry = "3b"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': 'experiment'})

In [16]:
df = df.dropna(axis=1, how='all')
df.columns
df.dropna(subset=['ape'], inplace=True)
df.rename(columns={"ape": "participant",
    "vector/innerb":"vector_innerb",
    "beacon/outerb":"beacon_outerb",
    'landmark area':"landmark_area",
    'rest area':"rest_area"}, inplace=True)

replace_list = ['experiment_name', 'condition']
for x in replace_list:
    df[x].replace(' ', '_', inplace=True, regex=True)
df['condition'].replace('ctrl_all_','ctrl_all', inplace=True, regex=True)

In [17]:
# df.columns

date_list = [['1b','2007'],
             ['2b','2007'],
             ['2c','2007'],
             ['3b','2008']]
for x,y in date_list:
    df.loc[df.experiment == x, ['year']] = y

In [18]:
df=df[[ 'study_id', 'experiment', 'experiment_name','year', 'participant', 'sex', 'species','condition',
         'a1', 'a2', 'a3',
       'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6',
       'b7', 'b8', 'b9', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9',
       'd1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8', 'd9', 'e1', 'e2', 'e3',
       'e4', 'e5', 'e6', 'e7', 'e8', 'e9', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6',
       'f7', 'f8', 'f9', 'g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9',
       'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'h8', 'h9', 'i1', 'i2', 'i3',
       'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'center', 'vector_innerb',
       'beacon_outerb', 'landmark_area', 'rest_area']]



In [19]:
exp1 = df[df['experiment'] == '1b'] 
exp2 = df[df['experiment'] == '2b']
exp3 = df[df['experiment'] == '2c']
exp4 = df[df['experiment'] == '3b']

experiments = [[exp1, 'poti2010searching_exp1b'], ##connects df with name of output dataset
                [ exp2, 'poti2010searching_exp2b'],
                [exp3, 'poti2010searching_exp2c'], 
                [exp4, 'poti2010searching_exp3b']]

for x,y in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [20]:
# for index in range(1,5):
#     exp = df[df['experiment'] == str(index)]
#     exp = exp.dropna(axis=1, how='all')
#     comp_out_path = os.path.join(out_pathway, 'poti2010searching_exp'+str(index)+'_summary_standardized.csv')
#     exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
#     names = exp.columns.tolist()
#     exp_g = pd.DataFrame(names)
#     exp_g = exp_g.rename(columns={0: "column_name"})
#     exp_g["description"] = ""
#     exp_g=exp_g[["column_name", "description"]]
#     comp_out_path_glossary = os.path.join(out_pathway, 'poti2010searching_exp'+str(index)+'_summary_glossary.csv')
#     exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)